# Simulating the transversal STAR architecture

In [ ]:
import tsim
import numpy as np
import pymatching
import matplotlib.pyplot as plt

Fault-tolerant single qubit rotations usually need to be compiled into the Clifford+T gateset. This makes rotations costly since a single rotation will require many T gates.

For small-angle rotations and target infidelities $>10^{-6}$, we can use the space-time efficient analog rotation (STAR) approach.

Here, we will follow the description in
**Transversal STAR architecture for megaquop-scale quantum simulation with neutral atoms**, Ismail et al (2025)
[arXiv:2509.18294](https://arxiv.org/abs/2509.18294).

The central idea behind STAR is remarkably simple. Instead of implementing a fully fault-tolerant logical rotation, we directly apply physical single-qubit rotations along the support of a logical operator of the code. Because this operation is not an exact logical gate, it generally takes the encoded state out of the code space.

However, after subsequent rounds of stabilizer measurements, the state is projected back into the code space. Depending on the measured syndrome outcomes, the resulting logical state may correspond to the desired logical rotation. In this way, STAR trades exact fault-tolerant synthesis for a probabilistic but highly resource-efficient analog implementation.

<img src="../star-arch.png" alt="arXiv:2509.18294" width=600/>

In [ ]:
def physical_angle(logical_angle: float, k: int) -> float:
    """
    Compute the physical rotation angle needed to achieve a logical rotation of angle `logical_angle` on `k` copies.
    Args:
        logical_angle (float): The logical rotation angle in radians.
        k (int): The number of copies.
    Returns:
        float: The physical rotation angle in radians.
    """

    assert k % 2 == 1 and k > 0, "k must be a positive odd integer"
    sign = -1 if (k + 1) % 4 == 0 else 1
    x = np.tan(logical_angle / 2) ** (1 / k)
    theta_phys = 2 * np.arctan(x)
    return sign * theta_phys

def load_circuit(logical_angle: float, distance: int) -> tsim.Circuit:
    with open(f"circuits/star_d={distance}.stim") as file:
        circuit_str = file.read()
        
    angle = physical_angle(logical_angle, min(distance, 7))
    circuit_str = circuit_str.replace("{physical_angle}", f"{angle / np.pi:.12f}")
    circuit_str = circuit_str.replace("{inverse_logical_angle}", f"{-logical_angle / np.pi:.12f}")
    return tsim.Circuit(circuit_str)

def plot(ler: dict[tuple[int, float], float], psr: dict[tuple[int, float], float]):
    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.0, 4), constrained_layout=True)

    distances: list[int] = sorted(list(set([d for d, _ in ler])))
    rotation_angles: list[float] = sorted(list(set([t for _, t in ler])))

    markers = ["o", "s", "^"]
    for distance, marker in zip(distances, markers, strict=False):
        ler_values = [ler[(distance, theta)] for theta in rotation_angles]
        psr_values = [psr[(distance, theta)] for theta in rotation_angles]

        ax1.loglog(rotation_angles, ler_values, marker=marker, markerfacecolor="white", label=rf"$d={distance}$")
        ax2.semilogx(rotation_angles, psr_values, marker=marker, markerfacecolor="white", label=rf"$d={distance}$")

    ax1.set_xlabel(r"Rotation angle $\theta$ (rad)")
    ax1.set_ylabel(r"Logical error rate / distance")
    ax1.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.5)
    ax1.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.25)
    ax1.legend(frameon=False, loc="best")

    ax2.set_xlabel(r"Rotation angle $\theta$ (rad)")
    ax2.set_ylabel("Post-selection rate")
    ax2.set_ylim(0, 1)
    ax2.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.5)
    ax2.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.25)
    ax2.legend(frameon=False, loc="best")

We begin by loading a pre-generated circuit containing multiple stabilizer readout rounds for a distance-5 rotated surface code. In Tick 14, single-qubit $Z$-rotations are applied along the support of a logical $Z$ operator. Because these rotations are applied transversally at the physical level, they do not correspond to an exact logical operation and temporarily take the state out of the code space.

The rotation step is followed by $d=5$ rounds of stabilizer measurements, which project the state back into the code space. Finally, we apply a noiseless logical unrotation and measure the logical $X$ operator. This final unrotation is included only for simulation purposes: in the absence of noise, the protocol should ideally return the logical $|+\rangle$ state with unit probability. Under noisy evolution, deviations from this outcome allow us to directly estimate the logical fidelity of the protocol.

In [ ]:
c = load_circuit(0.001, distance=5)
c.without_noise().diagram("timeslice-svg")

With Tsim, we can simulate the STAR protocol in a few lines of code for distance $3$ and $5$ surface codes and various rotation angles:

In [ ]:
rotation_angles = np.logspace(-4.5, -0.5, 5) * np.pi
distances = [3, 5]
logical_error_rate, post_selection_rate = {}, {}

for distance, theta in [(d, t) for d in distances for t in rotation_angles]:
    c = load_circuit(theta, distance)

    # Tsim simulation
    sampler = c.compile_detector_sampler(seed=0)
    dets, obs = sampler.sample(shots=300_000, separate_observables=True)

    # Post-processing (post-selection and decoding)
    selection = ~dets[:, :3 * (distance**2 - 1)].any(axis=1)
    dets, obs = dets[selection], obs[selection]
    model = c.detector_error_model()
    matching = pymatching.Matching.from_detector_error_model(model)
    flips = matching.decode_batch(dets)
    corrected_obs = np.logical_xor(obs, flips)

    logical_error_rate[(distance, theta)] = np.mean(corrected_obs) / distance
    post_selection_rate[(distance, theta)] = np.mean(selection)


This reproduces a key result from [arXiv:2509.18294](https://arxiv.org/abs/2509.18294). We observe that the logical error rate is initially suppressed for small rotation angles before eventually saturating at the baseline logical error rate of the surface code. The postselection probability depends on both the rotation angle and the code distance. For small angles, the success probability is approximately 80\% for $d=3$ and 50\% for $d=5$, and decreases as the rotation angle increases.

These simulations highlight an important opportunity for near-term quantum error correction protocols involving small-angle rotations, such as Trotterized Hamiltonian simulation. Rather than synthesizing fully fault-tolerant logical rotations using large numbers of distilled or cultivated T gates, the STAR approach enables the efficient preparation of small-angle rotational resource states without requiring magic-state factories and with only modest postselection overhead. These rotational states can then be injected or teleported into logical circuits to implement the desired rotations efficiently.

In [ ]:
plot(logical_error_rate, post_selection_rate)